# Data Quality Correction

Before performing EDA (Exploratory Data Analysis), some initial data cleaning is necessary so you can trust what you’re analyzing.

**Basic cleaning includes:**

- Loading the data correctly (e.g., proper parsing of dates, encoding, etc.)

- Checking for and handling:

    - Missing values (at least identifying them)

    - Incorrect data types (e.g., string instead of numeric)

    - Obvious outliers or garbage values (e.g., negative age)

    - Duplicate rows

> 🧼 This stage is often called "initial or light cleaning" — it helps avoid misleading results during EDA.

## Importing Libraries

In [1]:
from sklearn.pipeline import Pipeline
from src.analyze.data_correction.schema_correction import MissingValueNormalizer, FrequencyFilter
import pandas as pd
import cloudpickle
import warnings
import os

from src.analyze.data_inspection.inspector import InspectData
from src.analyze.data_correction.schema_correction import (ColumnNameCorrector, ValueCorrector)

warnings.filterwarnings("ignore")

In [2]:
os.getcwd()

'/Users/rahulshelke/Documents/Data-Science/Data-Science-Projects/heart-stroke-prediction'

In [3]:
%pwd

'/Users/rahulshelke/Documents/Data-Science/Data-Science-Projects/heart-stroke-prediction'

In [4]:
cd notebooks

/Users/rahulshelke/Documents/Data-Science/Data-Science-Projects/heart-stroke-prediction/notebooks


## **Setup Environment**

In [5]:
# # connect gdrive
# from google.colab import drive
# drive.mount('/content/drive')

# import sys
# import os

# # Define the path to your src folder
# # Note: "My Drive" contains a space, which is fine in a Python string
# path_to_src = '/content/drive/MyDrive/Data Science/Classification Problems/01 Heart Stroke Prediction/src'

# # Add to sys.path if it's not already there
# if path_to_src not in sys.path:
#     sys.path.append(path_to_src)

In [6]:
# # Change directory to where setup.py is located
# %cd "/content/drive/MyDrive/Data Science/Classification Problems/01 Heart Stroke Prediction/"

# # Install the current directory in "editable" mode (-e)
# !pip install -e .

In [7]:
DATA_FILES = os.listdir(os.path.join(os.getcwd(), "data"))
DATA_FILES

['heart_stroke_data.csv',
 'healthcare-dataset-stroke-data.csv',
 'heart_stroke_feature_engineered_data.csv']

In [8]:
ARTIFACT_FILES = os.listdir(os.path.join(os.getcwd(), "artifacts"))
ARTIFACT_FILES

['missing_imputer_validation_pipeline.pkl',
 'data_correction_pipeline.pkl',
 'feature_engineering_pipeline_v1.pkl',
 'missing_imputer_pipeline.pkl',
 'outlier_handler_pipeline.pkl']

In [9]:
DATA_FILE_NAME = {
    'raw': 'healthcare-dataset-stroke-data.csv',
    'clean': 'heart_stroke_data.csv', 
}

ARTIFACT_FILE_NAME = {
    'data_correct': 'data_correction_pipeline.pkl',
}

## **Import the CSV Data as Pandas DataFrame**

In [10]:
os.getcwd()

'/Users/rahulshelke/Documents/Data-Science/Data-Science-Projects/heart-stroke-prediction/notebooks'

In [11]:
df = pd.read_csv(os.path.join(os.getcwd(), "data", DATA_FILE_NAME['raw']))

In [12]:
df.shape

(5110, 12)

In [13]:
df.head()

,id,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,9046,Male,67.0,0,1,Yes,Private,Urban,228.69,36.6,formerly smoked,1
1,51676,Female,61.0,0,0,Yes,Self-employed,Rural,202.21,NaN,never smoked,1
2,31112,Male,80.0,0,1,Yes,Private,Rural,105.92,32.5,never smoked,1
3,60182,Female,49.0,0,0,Yes,Private,Urban,171.23,34.4,smokes,1
4,1665,Female,79.0,1,0,Yes,Self-employed,Rural,174.12,24.0,never smoked,1


In [14]:
df['stroke'].value_counts()

stroke
0    4861
1     249
Name: count, dtype: int64

In [15]:
df['stroke'].value_counts(normalize=True)

stroke
0    0.951272
1    0.048728
Name: proportion, dtype: float64

## Check Schema, shape, columns

In [16]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5110 entries, 0 to 5109
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id                 5110 non-null   int64  
 1   gender             5110 non-null   object 
 2   age                5110 non-null   float64
 3   hypertension       5110 non-null   int64  
 4   heart_disease      5110 non-null   int64  
 5   ever_married       5110 non-null   object 
 6   work_type          5110 non-null   object 
 7   Residence_type     5110 non-null   object 
 8   avg_glucose_level  5110 non-null   float64
 9   bmi                4909 non-null   float64
 10  smoking_status     5110 non-null   object 
 11  stroke             5110 non-null   int64  
dtypes: float64(3), int64(4), object(5)
memory usage: 479.2+ KB


- **~ 5k rows**
- **12 columns** (including target column)

## Standardize Column Names

In [17]:
df.columns

Index(['id', 'gender', 'age', 'hypertension', 'heart_disease', 'ever_married',
       'work_type', 'Residence_type', 'avg_glucose_level', 'bmi',
       'smoking_status', 'stroke'],
      dtype='object')

- will keep all to lower case.

In [18]:
column_correction_dict = {
    'Residence_type': 'residence_type'
}

In [19]:
column_corrector = ColumnNameCorrector(column_correction_dict)

In [20]:
df = column_corrector.transform(df)

In [21]:
df.head()

,id,gender,age,hypertension,heart_disease,ever_married,work_type,residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,9046,Male,67.0,0,1,Yes,Private,Urban,228.69,36.6,formerly smoked,1
1,51676,Female,61.0,0,0,Yes,Self-employed,Rural,202.21,NaN,never smoked,1
2,31112,Male,80.0,0,1,Yes,Private,Rural,105.92,32.5,never smoked,1
3,60182,Female,49.0,0,0,Yes,Private,Urban,171.23,34.4,smokes,1
4,1665,Female,79.0,1,0,Yes,Self-employed,Rural,174.12,24.0,never smoked,1


## Fix Categorical Values (String Standardization)

In [22]:
inspector = InspectData(df)

In [23]:
df.columns

Index(['id', 'gender', 'age', 'hypertension', 'heart_disease', 'ever_married',
       'work_type', 'residence_type', 'avg_glucose_level', 'bmi',
       'smoking_status', 'stroke'],
      dtype='object')

In [24]:
cat_col = ['gender', 'hypertension', 'heart_disease', 'ever_married',
       'work_type', 'residence_type', 'smoking_status', 'stroke']

In [25]:
inspector.categorical_column_inspect(cat_col)

-------------------------

column name: gender

# unique: 3

unique values: ['Male' 'Female' 'Other']

max: Female

min: Other

null count: 0

-------------------------

column name: hypertension

# unique: 2

unique values: [0 1]

max: 0

min: 1

null count: 0

-------------------------

column name: heart_disease

# unique: 2

unique values: [1 0]

max: 0

min: 1

null count: 0

-------------------------

column name: ever_married

# unique: 2

unique values: ['Yes' 'No']

max: Yes

min: No

null count: 0

-------------------------

column name: work_type

# unique: 5

unique values: ['Private' 'Self-employed' 'Govt_job' 'children' 'Never_worked']

max: Private

min: Never_worked

null count: 0

-------------------------

column name: residence_type

# unique: 2

unique values: ['Urban' 'Rural']

max: Urban

min: Rural

null count: 0

-------------------------

column name: smoking_status

# unique: 4

unique values: ['formerly smoked' 'never smoked' 'smokes' 'Unknown']

max: never s

**Correction needed to this columns:**
- gender
- ever_married
- work_type
- residence_type
- smoking_status

In [26]:
value_correction_dict = {
    'gender': {'Male': 'male', 'Female': 'female', 'Other': 'other'},
    'ever_married': {'Yes':'yes', 'No': 'no'},
    'residence_type': {'Urban': 'urban', 'Rural': 'rural'},
    'smoking_status': {'formerly smoked': 'formerly_smoked', 
                            'never smoked': 'never_smoked', 
                            'Unknown': 'unknown'},
    'work_type': {'Private':'private', 'self-employed': 'self_employed', 
                            'Govt_job': 'govt_job', 'Never_worked': 'never_worked',
                            'children': 'children'},
}

In [27]:
value_corrector = ValueCorrector(column_mappings=value_correction_dict)

In [28]:
df = value_corrector.transform(X=df)

In [29]:
df.head()

,id,gender,age,hypertension,heart_disease,ever_married,work_type,residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,9046,male,67.0,0,1,yes,private,urban,228.69,36.6,formerly_smoked,1
1,51676,female,61.0,0,0,yes,self_employed,rural,202.21,NaN,never_smoked,1
2,31112,male,80.0,0,1,yes,private,rural,105.92,32.5,never_smoked,1
3,60182,female,49.0,0,0,yes,private,urban,171.23,34.4,smokes,1
4,1665,female,79.0,1,0,yes,self_employed,rural,174.12,24.0,never_smoked,1


In [30]:
df.tail()

,id,gender,age,hypertension,heart_disease,ever_married,work_type,residence_type,avg_glucose_level,bmi,smoking_status,stroke
5105,18234,female,80.0,1,0,yes,private,urban,83.75,NaN,never_smoked,0
5106,44873,female,81.0,0,0,yes,self_employed,urban,125.20,40.0,never_smoked,0
5107,19723,female,35.0,0,0,yes,self_employed,rural,82.99,30.6,never_smoked,0
5108,37544,male,51.0,0,0,yes,private,rural,166.29,25.6,formerly_smoked,0
5109,44679,female,44.0,0,0,yes,govt_job,urban,85.28,26.2,unknown,0


## Fix Categorical Values Frequency (Handling category frequency)

In [31]:
frequency_filter = FrequencyFilter(threshold=1, action='drop')

In [32]:
df = frequency_filter.fit_transform(df)

In [33]:
df.shape

(5109, 12)

In [34]:
df[df['gender'] == 'other']

,id,gender,age,hypertension,heart_disease,ever_married,work_type,residence_type,avg_glucose_level,bmi,smoking_status,stroke


In [35]:
df['gender'].value_counts()

gender
female    2994
male      2115
Name: count, dtype: int64

## Fix Numerical Values (Context Standardization)

In [36]:
num_col = ['age', 'avg_glucose_level', 'bmi']

In [37]:
inspector.numerical_column_inspect(num_col)

-------------------------

column name: age

# unique: 104

+ve values: True | (104)

zeros: 0

-ve values: False | (0)

null values: 0

-------------------------

column name: avg_glucose_level

# unique: 3979

+ve values: True | (3979)

zeros: 0

-ve values: False | (0)

null values: 0

-------------------------

column name: bmi

# unique: 418

+ve values: True | (418)

zeros: 0

-ve values: False | (0)

null values: 201

-------------------------


**bmi**

- `bmi` feature has 4% (201) values missing
- replacing `nan` with -1

⚠️ **Unusual / Suspicious Values:**

| Range                               | Possible Issue                                                   |
| ----------------------------------- | ---------------------------------------------------------------- |
| **< 40**                            | Rare, possible hypoglycemia or data error                        |
| **> 250–300**                       | Very high, may indicate poorly controlled diabetes or unit error |
| **> 600**                           | Extremely rare, possibly a data or entry error                   |
| **Unrealistic values (e.g. >1000)** | Almost certainly data error                                      |


## Remove Unwanted/Corrupted Columns/Rows

In [38]:
df[df.duplicated()]

,id,gender,age,hypertension,heart_disease,ever_married,work_type,residence_type,avg_glucose_level,bmi,smoking_status,stroke


In [39]:
df[df.drop(columns="id", axis=1).duplicated()]

,id,gender,age,hypertension,heart_disease,ever_married,work_type,residence_type,avg_glucose_level,bmi,smoking_status,stroke


## Identifying missing values (off context)

In [40]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 5109 entries, 0 to 5109
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id                 5109 non-null   int64  
 1   gender             5109 non-null   object 
 2   age                5109 non-null   float64
 3   hypertension       5109 non-null   int64  
 4   heart_disease      5109 non-null   int64  
 5   ever_married       5109 non-null   object 
 6   work_type          5109 non-null   object 
 7   residence_type     5109 non-null   object 
 8   avg_glucose_level  5109 non-null   float64
 9   bmi                4908 non-null   float64
 10  smoking_status     5109 non-null   object 
 11  stroke             5109 non-null   int64  
dtypes: float64(3), int64(4), object(5)
memory usage: 518.9+ KB


In [41]:
df.isna().sum()

id                     0
gender                 0
age                    0
hypertension           0
heart_disease          0
ever_married           0
work_type              0
residence_type         0
avg_glucose_level      0
bmi                  201
smoking_status         0
stroke                 0
dtype: int64

## Pipeline: Basic Cleaning

### 1. Build pipeline

In [43]:
# Create pipeline
data_correction_pipeline = Pipeline(
    steps=[
        ('correct_column_names', ColumnNameCorrector(mapping=column_correction_dict)),
        ('correct_column_values', ValueCorrector(column_mappings=value_correction_dict)),
        ('missing_normalizer', MissingValueNormalizer(missing_tokens=['unknown'])),
        ('frequency_filter', FrequencyFilter(threshold=1, action='drop')),
    ]
)

In [44]:
data_correction_pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('correct_column_names', ...), ('correct_column_values', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,mapping,{'Residence_type': 'residence_type'}
,column_mappings,"{'ever_married': {'No': 'no', 'Yes': 'yes'}, 'gender': {'Female': 'female', 'Male': 'male', 'Other': 'other'}, 'residence_type': {'Rural': 'rural', 'Urban': 'urban'}, 'smoking_status': {'Unknown': 'unknown', 'formerly smoked': 'formerly_smoked', 'never smoked': 'never_smoked'}, ...}"
,missing_tokens,['unknown']
,threshold,1
,action,'drop'
,fill_value,'other'


#### data-head

In [45]:
df.head()

,id,gender,age,hypertension,heart_disease,ever_married,work_type,residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,9046,male,67.0,0,1,yes,private,urban,228.69,36.6,formerly_smoked,1
1,51676,female,61.0,0,0,yes,self_employed,rural,202.21,NaN,never_smoked,1
2,31112,male,80.0,0,1,yes,private,rural,105.92,32.5,never_smoked,1
3,60182,female,49.0,0,0,yes,private,urban,171.23,34.4,smokes,1
4,1665,female,79.0,1,0,yes,self_employed,rural,174.12,24.0,never_smoked,1


In [46]:
data_correction_pipeline.fit_transform(df).head()

,id,gender,age,hypertension,heart_disease,ever_married,work_type,residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,9046,male,67.0,0,1,yes,private,urban,228.69,36.6,formerly_smoked,1
1,51676,female,61.0,0,0,yes,self_employed,rural,202.21,NaN,never_smoked,1
2,31112,male,80.0,0,1,yes,private,rural,105.92,32.5,never_smoked,1
3,60182,female,49.0,0,0,yes,private,urban,171.23,34.4,smokes,1
4,1665,female,79.0,1,0,yes,self_employed,rural,174.12,24.0,never_smoked,1


#### data-tail

In [47]:
df.tail()

,id,gender,age,hypertension,heart_disease,ever_married,work_type,residence_type,avg_glucose_level,bmi,smoking_status,stroke
5105,18234,female,80.0,1,0,yes,private,urban,83.75,NaN,never_smoked,0
5106,44873,female,81.0,0,0,yes,self_employed,urban,125.20,40.0,never_smoked,0
5107,19723,female,35.0,0,0,yes,self_employed,rural,82.99,30.6,never_smoked,0
5108,37544,male,51.0,0,0,yes,private,rural,166.29,25.6,formerly_smoked,0
5109,44679,female,44.0,0,0,yes,govt_job,urban,85.28,26.2,unknown,0


In [48]:
data_correction_pipeline.fit_transform(df).tail()

,id,gender,age,hypertension,heart_disease,ever_married,work_type,residence_type,avg_glucose_level,bmi,smoking_status,stroke
5105,18234,female,80.0,1,0,yes,private,urban,83.75,NaN,never_smoked,0
5106,44873,female,81.0,0,0,yes,self_employed,urban,125.20,40.0,never_smoked,0
5107,19723,female,35.0,0,0,yes,self_employed,rural,82.99,30.6,never_smoked,0
5108,37544,male,51.0,0,0,yes,private,rural,166.29,25.6,formerly_smoked,0
5109,44679,female,44.0,0,0,yes,govt_job,urban,85.28,26.2,NaN,0


### 2. Apply pipeline

In [49]:
df = pd.read_csv("data/healthcare-dataset-stroke-data.csv")

# df.head()

In [50]:
df.isna().sum()

id                     0
gender                 0
age                    0
hypertension           0
heart_disease          0
ever_married           0
work_type              0
Residence_type         0
avg_glucose_level      0
bmi                  201
smoking_status         0
stroke                 0
dtype: int64

In [51]:
df_corrected = data_correction_pipeline.fit_transform(df)

In [52]:
df_corrected.head()

,id,gender,age,hypertension,heart_disease,ever_married,work_type,residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,9046,male,67.0,0,1,yes,private,urban,228.69,36.6,formerly_smoked,1
1,51676,female,61.0,0,0,yes,self_employed,rural,202.21,NaN,never_smoked,1
2,31112,male,80.0,0,1,yes,private,rural,105.92,32.5,never_smoked,1
3,60182,female,49.0,0,0,yes,private,urban,171.23,34.4,smokes,1
4,1665,female,79.0,1,0,yes,self_employed,rural,174.12,24.0,never_smoked,1


In [53]:
df_corrected.tail()

,id,gender,age,hypertension,heart_disease,ever_married,work_type,residence_type,avg_glucose_level,bmi,smoking_status,stroke
5105,18234,female,80.0,1,0,yes,private,urban,83.75,NaN,never_smoked,0
5106,44873,female,81.0,0,0,yes,self_employed,urban,125.20,40.0,never_smoked,0
5107,19723,female,35.0,0,0,yes,self_employed,rural,82.99,30.6,never_smoked,0
5108,37544,male,51.0,0,0,yes,private,rural,166.29,25.6,formerly_smoked,0
5109,44679,female,44.0,0,0,yes,govt_job,urban,85.28,26.2,NaN,0


In [54]:
df_corrected.isna().sum()

id                      0
gender                  0
age                     0
hypertension            0
heart_disease           0
ever_married            0
work_type               0
residence_type          0
avg_glucose_level       0
bmi                   201
smoking_status       1544
stroke                  0
dtype: int64

### 3. Save pipeline & Data

In [55]:
with open(os.path.join(os.getcwd(), "artifacts", ARTIFACT_FILE_NAME["data_correct"]), "wb" ) as f:
    cloudpickle.dump(data_correction_pipeline, f)

In [56]:
with open(os.path.join(os.getcwd(), "artifacts", ARTIFACT_FILE_NAME["data_correct"]), "rb" ) as f:
    data_correction_pipeline = cloudpickle.load(f)

In [57]:
data_correction_pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('correct_column_names', ...), ('correct_column_values', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,mapping,{'Residence_type': 'residence_type'}
,column_mappings,"{'ever_married': {'No': 'no', 'Yes': 'yes'}, 'gender': {'Female': 'female', 'Male': 'male', 'Other': 'other'}, 'residence_type': {'Rural': 'rural', 'Urban': 'urban'}, 'smoking_status': {'Unknown': 'unknown', 'formerly smoked': 'formerly_smoked', 'never smoked': 'never_smoked'}, ...}"
,missing_tokens,['unknown']
,threshold,1
,action,'drop'
,fill_value,'other'


In [58]:
df_corrected.to_csv(os.path.join(os.getcwd(), "data", DATA_FILE_NAME["clean"]), index=False)

## Dummy Data

In [61]:
df_corrected.shape

(5109, 12)

In [62]:
from sklearn.model_selection import train_test_split

In [68]:
train, test = train_test_split(df_corrected, test_size=0.025, stratify=df_corrected['stroke'])

In [69]:
test

,id,gender,age,hypertension,heart_disease,ever_married,work_type,residence_type,avg_glucose_level,bmi,smoking_status,stroke
4610,72725,female,26.0,0,0,no,govt_job,urban,59.67,24.5,smokes,0
3793,52668,female,24.0,0,0,no,private,urban,65.44,23.6,never_smoked,0
3678,47784,female,5.0,0,0,no,children,rural,123.49,19.5,NaN,0
2055,31746,female,62.0,0,0,yes,private,rural,83.85,24.5,never_smoked,0
1266,49412,male,63.0,0,0,yes,govt_job,urban,66.13,46.2,never_smoked,0
...,...,...,...,...,...,...,...,...,...,...,...,...
4359,10135,female,37.0,0,0,no,private,rural,112.02,29.1,NaN,0
4492,34045,female,8.0,0,0,no,children,urban,87.15,16.1,NaN,0
1760,69524,male,56.0,0,0,yes,self_employed,urban,94.07,31.5,never_smoked,0
2042,47701,male,8.0,0,0,no,children,urban,104.51,20.6,NaN,0


In [72]:
test['smoking_status'].value_counts()

smoking_status
never_smoked       50
smokes             24
formerly_smoked    19
Name: count, dtype: int64

### **Next Action:**

- Handle Missing Values
- Handle Outliers